In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_sub = pd.read_csv('../data/sample_submission.csv')

target = 'Irrigation_Need'
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
rev_target_map = {v: k for k, v in target_map.items()}

train[target] = train[target].map(target_map)

In [3]:
def extract_logic_features(df):
    df['soil_lt_25'] = (df['Soil_Moisture'] < 25).astype(int)
    df['rain_lt_300'] = (df['Rainfall_mm'] < 300).astype(int)
    df['temp_gt_30'] = (df['Temperature_C'] > 30).astype(int)
    df['wind_gt_10'] = (df['Wind_Speed_kmh'] > 10).astype(int)
    
    df['is_harvest'] = (df['Crop_Growth_Stage'] == 'Harvest').astype(int)
    df['is_sowing'] = (df['Crop_Growth_Stage'] == 'Sowing').astype(int)
    df['mulching_yes'] = (df['Mulching_Used'] == 'Yes').astype(int)
    
    # 模拟 Simple Formula 分数
    df['high_score'] = 2*df['soil_lt_25'] + 2*df['rain_lt_300'] + df['temp_gt_30'] + df['wind_gt_10']
    df['low_score'] = 2*df['is_harvest'] + 2*df['is_sowing'] + df['mulching_yes']
    df['logic_score'] = df['high_score'] - df['low_score']
    
    return df

train = extract_logic_features(train)
test = extract_logic_features(test)

# 处理其余类别特征
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

In [4]:
import optuna

def objective(trial):
    X = train.drop(['id', 'Irrigation_Need'], axis=1)
    y = train['Irrigation_Need']
    
    # 搜索空间定义
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1200),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        # 增加正则化，破除 0.97 的关键
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        
        'objective': 'multi:softprob',
        'num_class': 3,
        'tree_method': 'hist',
        'device': 'cuda',
        'enable_categorical': True,
        'random_state': 42,
        'n_jobs': -1
    }
    
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train)
        
        preds = model.predict(X_val)
        cv_scores.append(accuracy_score(y_val, preds))
        
    return np.mean(cv_scores)

In [6]:
import logging
optuna.logging.set_verbosity(optuna.logging.INFO)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(f"Best Accuracy: {study.best_value}")
print(f"Best Params: {study.best_params}")

[I 2026-04-07 20:29:42,402] A new study created in memory with name: no-name-495d3040-8adb-4b76-8c50-4d9ecc5ed16b
[I 2026-04-07 20:31:19,567] Trial 0 finished with value: 0.9855174603174603 and parameters: {'n_estimators': 605, 'max_depth': 8, 'learning_rate': 0.0377009294206707, 'subsample': 0.6252463345881776, 'colsample_bytree': 0.9711676462472439, 'reg_alpha': 0.5479831333462571, 'reg_lambda': 0.0017704334086040474, 'min_child_weight': 3}. Best is trial 0 with value: 0.9855174603174603.
[I 2026-04-07 20:32:06,566] Trial 1 finished with value: 0.9852396825396825 and parameters: {'n_estimators': 738, 'max_depth': 3, 'learning_rate': 0.01842425550570268, 'subsample': 0.8830663195686852, 'colsample_bytree': 0.7274834548214045, 'reg_alpha': 0.1486118966513021, 'reg_lambda': 3.821551762251233, 'min_child_weight': 9}. Best is trial 0 with value: 0.9855174603174603.
[I 2026-04-07 20:33:41,291] Trial 2 finished with value: 0.9855301587301588 and parameters: {'n_estimators': 791, 'max_depth'

KeyboardInterrupt: 

In [7]:
# 1. 提取 Trial 11 的最佳参数
best_params = {
    'n_estimators': 795,
    'max_depth': 7,
    'learning_rate': 0.0513701122723203,
    'subsample': 0.7496941785153926,
    'colsample_bytree': 0.9820097767370177,
    'reg_alpha': 0.01731925241887879,
    'reg_lambda': 0.06097707746058942,
    'min_child_weight': 3,
    'objective': 'multi:softprob',
    'num_class': 3,
    'tree_method': 'hist', # 确保使用之前的配置
    'device': 'cuda',
    'enable_categorical': True,
    'random_state': 42
}

# 2. 全量拟合
X_all = train.drop(['id', 'Irrigation_Need'], axis=1)
y_all = train['Irrigation_Need']
X_test = test.drop(['id'], axis=1)

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_all, y_all)

# 3. 生成提交文件
test_probs = final_model.predict_proba(X_test)
test_preds = np.argmax(test_probs, axis=1)
sample_sub['Irrigation_Need'] = [rev_target_map[p] for p in test_preds]
sample_sub.to_csv('../submissions/optuna_trial_11.csv', index=False)

print("Optuna optimized submission saved! Go get that 0.97+!")

Optuna optimized submission saved! Go get that 0.97+!


In [9]:
print(train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 31 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   id                       630000 non-null  int64   
 1   Soil_Type                630000 non-null  category
 2   Soil_pH                  630000 non-null  float64 
 3   Soil_Moisture            630000 non-null  float64 
 4   Organic_Carbon           630000 non-null  float64 
 5   Electrical_Conductivity  630000 non-null  float64 
 6   Temperature_C            630000 non-null  float64 
 7   Humidity                 630000 non-null  float64 
 8   Rainfall_mm              630000 non-null  float64 
 9   Sunlight_Hours           630000 non-null  float64 
 10  Wind_Speed_kmh           630000 non-null  float64 
 11  Crop_Type                630000 non-null  category
 12  Crop_Growth_Stage        630000 non-null  category
 13  Season                   630000 non-null  ca

In [11]:
# 暴力稳健版参数 - 专为破 0.97 设计
robust_params = {
    'n_estimators': 800,
    'max_depth': 4,              # 关键：树深砍半，强制泛化
    'learning_rate': 0.03,
    'reg_alpha': 0.5,            # 适度惩罚
    'reg_lambda': 5.0,           # 强力平滑逻辑边界
    'min_child_weight': 20,      # 强制每个分支必须有足够样本，过滤公式噪音
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'device': 'cuda',
    'enable_categorical': True,
    'random_state': 42
}

# 重新训练全量模型
model_robust = xgb.XGBClassifier(**robust_params)
model_robust.fit(X_all, y_all)

# 生成新预测
test_probs_robust = model_robust.predict_proba(X_test)
test_preds = np.argmax(test_probs_robust, axis=1)
sample_sub['Irrigation_Need'] = [rev_target_map[p] for p in test_preds]
sample_sub.to_csv('../submissions/robust.csv', index=False)
print('Done')

Done
